In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

DEAM_PATH = Path("../data/music/DEAM")

AUDIO_PATH = DEAM_PATH / "DEAM_audio" / "MEMD_audio"

ANNOTATIONS_PATH = (
    DEAM_PATH
    / "DEAM_Annotations"
    / "annotations"
    / "annotations per each rater"
    / "dynamic (per second annotations)"
)

VALENCE_PATH = ANNOTATIONS_PATH / "valence"
AROUSAL_PATH = ANNOTATIONS_PATH / "arousal"

audio_files = sorted(AUDIO_PATH.glob("*.mp3"))

records = []
trajectories = []

for audio_file in audio_files:

    song_id = int(audio_file.stem)

    valence_file = VALENCE_PATH / f"{song_id}.csv"
    arousal_file = AROUSAL_PATH / f"{song_id}.csv"

    if not valence_file.exists() or not arousal_file.exists():
        continue

    valence = pd.read_csv(valence_file)
    arousal = pd.read_csv(arousal_file)

    valence_columns = [
        column for column in valence.columns
        if column.startswith("sample_")
    ]

    arousal_columns = [
        column for column in arousal.columns
        if column.startswith("sample_")
    ]

    valence_values = valence[valence_columns].apply(
        pd.to_numeric,
        errors="coerce"
    )

    arousal_values = arousal[arousal_columns].apply(
        pd.to_numeric,
        errors="coerce"
    )

    valence_consensus = valence_values.mean(axis=0)
    arousal_consensus = arousal_values.mean(axis=0)

    valence_array = valence_consensus.to_numpy(dtype=float)
    arousal_array = arousal_consensus.to_numpy(dtype=float)

    records.append({
        "song_id": song_id,
        "audio_path": str(audio_file),
        "valence_mean": np.mean(valence_array),
        "valence_std": np.std(valence_array),
        "valence_min": np.min(valence_array),
        "valence_max": np.max(valence_array),
        "valence_range": np.ptp(valence_array),
        "arousal_mean": np.mean(arousal_array),
        "arousal_std": np.std(arousal_array),
        "arousal_min": np.min(arousal_array),
        "arousal_max": np.max(arousal_array),
        "arousal_range": np.ptp(arousal_array)
    })

    timestamps = [
        int(column.replace("sample_", "").replace("ms", ""))
        for column in valence_columns
    ]

    for timestamp, v, a in zip(
        timestamps,
        valence_array,
        arousal_array
    ):
        trajectories.append({
            "song_id": song_id,
            "timestamp_ms": timestamp,
            "valence": v,
            "arousal": a
        })

music_metadata = pd.DataFrame(records)
music_trajectories = pd.DataFrame(trajectories)

metadata_path = Path("../data/metadata")
metadata_path.mkdir(parents=True, exist_ok=True)

music_metadata.to_csv(
    metadata_path / "music_metadata.csv",
    index=False
)

music_trajectories.to_csv(
    metadata_path / "music_emotion_trajectories.csv",
    index=False
)

print("=" * 50)
print("DEAM metadata processing complete")
print("=" * 50)

print("Audio files found:", len(audio_files))
print("Songs processed:", len(music_metadata))
print("Trajectory rows:", len(music_trajectories))

print("\nMetadata:")
print(music_metadata.head())

print("\nMissing values:")
print(music_metadata.isna().sum())

DEAM metadata processing complete
Audio files found: 1802
Songs processed: 1802
Trajectory rows: 129995

Metadata:
   song_id                                         audio_path  valence_mean  \
0       10    ..\data\music\DEAM\DEAM_audio\MEMD_audio\10.mp3      0.043041   
1     1000  ..\data\music\DEAM\DEAM_audio\MEMD_audio\1000.mp3      0.327503   
2     1001  ..\data\music\DEAM\DEAM_audio\MEMD_audio\1001.mp3     -0.001067   
3     1002  ..\data\music\DEAM\DEAM_audio\MEMD_audio\1002.mp3     -0.047283   
4     1003  ..\data\music\DEAM\DEAM_audio\MEMD_audio\1003.mp3      0.175267   

   valence_std  valence_min  valence_max  valence_range  arousal_mean  \
0     0.053411    -0.015276     0.106513       0.121789     -0.171351   
1     0.031674     0.276816     0.371982       0.095166      0.481244   
2     0.016512    -0.038000     0.025000       0.063000      0.205650   
3     0.030267    -0.098000     0.055000       0.153000     -0.019483   
4     0.027095     0.140000     0.233000     

In [3]:
print("Songs:", len(music_metadata))
print("Unique song IDs:", music_metadata["song_id"].nunique())

print("\nMissing values:")
print(music_metadata.isna().sum())

print("\nDuplicate song IDs:")
print(music_metadata["song_id"].duplicated().sum())

print("\nValence range:")
print(
    music_metadata[["valence_min", "valence_max", "valence_mean"]].describe()
)

print("\nArousal range:")
print(
    music_metadata[["arousal_min", "arousal_max", "arousal_mean"]].describe()
)

Songs: 1802
Unique song IDs: 1802

Missing values:
song_id          0
audio_path       0
valence_mean     0
valence_std      0
valence_min      0
valence_max      0
valence_range    0
arousal_mean     0
arousal_std      0
arousal_min      0
arousal_max      0
arousal_range    0
dtype: int64

Duplicate song IDs:
0

Valence range:
       valence_min  valence_max  valence_mean
count  1802.000000  1802.000000   1802.000000
mean      0.033597     0.158987      0.097688
std       0.240152     0.227178      0.234633
min      -0.832000    -0.569400     -0.637343
25%      -0.130791     0.002162     -0.064162
50%       0.041962     0.173000      0.109598
75%       0.219750     0.330456      0.279371
max       0.564000     0.716000      0.637600

Arousal range:
       arousal_min  arousal_max  arousal_mean
count  1802.000000  1802.000000   1802.000000
mean      0.057858     0.206280      0.137346
std       0.285064     0.280805      0.279826
min      -0.772351    -0.654305     -0.673034
25%      

In [4]:
print("\nTrajectory song IDs:", music_trajectories["song_id"].nunique())

print("\nTrajectory missing values:")
print(music_trajectories.isna().sum())

print("\nSamples per song:")
print(
    music_trajectories
    .groupby("song_id")
    .size()
    .describe()
)


Trajectory song IDs: 1802

Trajectory missing values:
song_id         0
timestamp_ms    0
valence         0
arousal         0
dtype: int64

Samples per song:
count    1802.000000
mean       72.139290
std        76.582929
min        60.000000
25%        60.000000
50%        60.000000
75%        60.000000
max      1223.000000
dtype: float64


In [8]:
import sys
from pathlib import Path

sys.path.append("..")

from muext import process_music_files
music_features, music_errors = process_music_files(
    max_workers=8,
    limit=None
)

Starting music feature extraction
Total files : 1802
Workers     : 8

Processed 25/1802
Processed 50/1802
Processed 75/1802
Processed 100/1802
Processed 125/1802
Processed 150/1802
Processed 175/1802
Processed 200/1802
Processed 225/1802
Processed 250/1802
Processed 275/1802
Processed 300/1802
Processed 325/1802
Processed 350/1802
Processed 375/1802
Processed 400/1802
Processed 425/1802
Processed 450/1802
Processed 475/1802
Processed 500/1802
Processed 525/1802
Processed 550/1802
Processed 575/1802
Processed 600/1802
Processed 625/1802
Processed 650/1802
Processed 675/1802
Processed 700/1802
Processed 725/1802
Processed 750/1802
Processed 775/1802
Processed 800/1802
Processed 825/1802
Processed 850/1802
Processed 875/1802
Processed 900/1802
Processed 925/1802
Processed 950/1802
Processed 975/1802
Processed 1000/1802
Processed 1025/1802
Processed 1050/1802
Processed 1075/1802
Processed 1100/1802
Processed 1125/1802
Processed 1150/1802
Processed 1175/1802
Processed 1200/1802
Processed 12

In [1]:
import pandas as pd

music = pd.read_csv("../data/processed/music_features.csv")

print("Shape:", music.shape)

print("\nUnique songs:")
print(music["song_id"].nunique())

print("\nDuplicate song IDs:")
print(music["song_id"].duplicated().sum())

print("\nMissing values:")
print(music.isna().sum().sum())

print("\nMissing values by column:")
print(music.isna().sum()[music.isna().sum() > 0])

print("\nValence statistics:")
print(
    music[
        ["valence_mean", "valence_std", "valence_min", "valence_max"]
    ].describe()
)

print("\nArousal statistics:")
print(
    music[
        ["arousal_mean", "arousal_std", "arousal_min", "arousal_max"]
    ].describe()
)

Shape: (1802, 195)

Unique songs:
1802

Duplicate song IDs:
0

Missing values:
0

Missing values by column:
Series([], dtype: int64)

Valence statistics:
       valence_mean  valence_std  valence_min  valence_max
count   1802.000000  1802.000000  1802.000000  1802.000000
mean       0.097688     0.033603     0.033597     0.158987
std        0.234633     0.025193     0.240152     0.227178
min       -0.637343     0.003982    -0.832000    -0.569400
25%       -0.064162     0.017232    -0.130791     0.002162
50%        0.109598     0.027068     0.041962     0.173000
75%        0.279371     0.041487     0.219750     0.330456
max        0.637600     0.236320     0.564000     0.716000

Arousal statistics:
       arousal_mean  arousal_std  arousal_min  arousal_max
count   1802.000000  1802.000000  1802.000000  1802.000000
mean       0.137346     0.040464     0.057858     0.206280
std        0.279826     0.036488     0.285064     0.280805
min       -0.673034     0.003730    -0.772351    -0.654305

In [2]:
speech = pd.read_csv(
    "../data/metadata/speech_features.csv"
)

print(speech.columns.tolist())
print("\nShape:", speech.shape)

['file_id', 'dataset', 'file', 'speaker', 'emotion', 'emotion_id', 'sample_rate', 'duration', 'pitch_mean', 'pitch_median', 'pitch_min', 'pitch_max', 'pitch_std', 'pitch_range', 'energy_mean', 'energy_std', 'energy_min', 'energy_max', 'energy_range', 'zcr_mean', 'zcr_std', 'zcr_min', 'zcr_max', 'jitter', 'shimmer', 'mfcc_1_mean', 'mfcc_1_std', 'mfcc_2_mean', 'mfcc_2_std', 'mfcc_3_mean', 'mfcc_3_std', 'mfcc_4_mean', 'mfcc_4_std', 'mfcc_5_mean', 'mfcc_5_std', 'mfcc_6_mean', 'mfcc_6_std', 'mfcc_7_mean', 'mfcc_7_std', 'mfcc_8_mean', 'mfcc_8_std', 'mfcc_9_mean', 'mfcc_9_std', 'mfcc_10_mean', 'mfcc_10_std', 'mfcc_11_mean', 'mfcc_11_std', 'mfcc_12_mean', 'mfcc_12_std', 'mfcc_13_mean', 'mfcc_13_std']

Shape: (10322, 51)


In [3]:
import pandas as pd

speech = pd.read_csv("../data/metadata/speech_features.csv")

print("Dataset distribution:")
print(speech["dataset"].value_counts())

print("\nEmotion distribution:")
print(speech["emotion"].value_counts())

print("\nEmotion IDs:")
print(
    speech[["emotion", "emotion_id"]]
    .drop_duplicates()
    .sort_values("emotion_id")
)

print("\nEmotion by dataset:")
print(
    pd.crosstab(
        speech["dataset"],
        speech["emotion"]
    )
)

Dataset distribution:
dataset
CREMA-D    7442
RAVDESS    2880
Name: count, dtype: int64

Emotion distribution:
emotion
angry        1655
disgust      1655
fearful      1655
happy        1655
sad          1655
neutral      1279
calm          384
surprised     384
Name: count, dtype: int64

Emotion IDs:
        emotion emotion_id
7442    neutral          1
7446       calm          2
7454      happy          3
7462        sad          4
7470      angry          5
7478    fearful          6
7486    disgust          7
7494  surprised          8
0         angry        ANG
1       disgust        DIS
2       fearful        FEA
3         happy        HAP
4       neutral        NEU
5           sad        SAD

Emotion by dataset:
emotion  angry  calm  disgust  fearful  happy  neutral   sad  surprised
dataset                                                                
CREMA-D   1271     0     1271     1271   1271     1087  1271          0
RAVDESS    384   384      384      384    384      192 